# 🌤️ Weather–Energy Correlation Analysis

Investigate how outdoor temperature, humidity, and wind speed relate to heat pump
energy consumption. Uses the Open-Meteo historical weather API paired with actual
sensor readings from the Aiven database.

**Key Question**: How strongly does outdoor temperature predict heat pump energy use?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, timezone, date
from sqlalchemy import func, text

from app.database.database import get_db_session
from app.database.models import Sensor, SensorReading, Location
from app.services.weather import fetch_historical

pd.set_option('display.max_columns', None)
print("✅ Imports ready")

✅ Imports ready


## 1. Load Energy Data

Load hourly energy deltas from the cumulative heat pump meters:
- **warmepumpe_Energie_sum** — electrical energy input (kWh)
- **idm_aero_hp_warmemenge_gesamt** — thermal energy output (kWh)

In [ ]:
LOOKBACK_DAYS = 90  # adjust as needed

with get_db_session() as db:
    cutoff = datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)

    def load_sensor_df(name):
        sensor = db.query(Sensor).filter(Sensor.name == name).first()
        if not sensor:
            raise ValueError(f"Sensor '{name}' not found")
        readings = (db.query(SensorReading.timestamp, SensorReading.value)
                    .filter(SensorReading.sensor_id == sensor.id,
                            SensorReading.timestamp >= cutoff)
                    .order_by(SensorReading.timestamp).all())
        df = pd.DataFrame(readings, columns=['timestamp', 'value'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
        return df

    df_elec = load_sensor_df('warmepumpe_Energie_sum')
    df_therm = load_sensor_df('idm_aero_hp_warmemenge_gesamt')

    # Get location coords for weather
    loc = db.query(Location).filter(Location.city == 'Bonn').first()
    LAT, LON = loc.latitude, loc.longitude
    loc_city = loc.city

print(f"Electrical readings: {len(df_elec):,}")
print(f"Thermal readings: {len(df_therm):,}")
print(f"Location: {loc_city} ({LAT}, {LON})")

/tmp/ipykernel_20041/1905877623.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)


2026-03-24 15:23:40,398 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-24 15:23:40,399 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:23:40,567 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-24 15:23:40,568 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:23:40,630 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-24 15:23:40,631 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:23:40,672 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:23:40,675 INFO sqlalchemy.engine.Engine SELECT api_sensors.id AS api_sensors_id, api_sensors.device_id AS api_sensors_device_id, api_sensors.name AS api_sensors_name, api_sensors.description AS api_sensors_description, api_sensors.sensor_type_id AS api_sensors_sensor_type_id, api_sensors.location_id AS api_sensors_location_id, api_sensors.manufacturer AS api_sensors_manufacturer, api_sensors.model AS api_sensors_model, api_sensors.firmware_version AS api_sensors_

DetachedInstanceError: Instance <Location at 0x78c987de5dc0> is not bound to a Session; attribute refresh operation cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)

## 2. Derive Hourly Energy Consumption

The meters are cumulative — compute hourly deltas by resampling and differencing.

In [ ]:
def hourly_deltas(df):
    """Convert cumulative meter readings to hourly consumption."""
    ts = df.set_index('timestamp').sort_index()
    hourly = ts['value'].resample('1h').last().interpolate()
    delta = hourly.diff().dropna()
    # Filter out negative deltas (meter resets) and extreme outliers
    delta = delta[(delta >= 0) & (delta < delta.quantile(0.99))]
    return delta.reset_index()

df_elec_h = hourly_deltas(df_elec).rename(columns={'value': 'elec_kwh'})
df_therm_h = hourly_deltas(df_therm).rename(columns={'value': 'therm_kwh'})

print(f"Hourly electrical points: {len(df_elec_h)}")
print(f"Hourly thermal points: {len(df_therm_h)}")
df_elec_h.head()

## 3. Fetch Historical Weather

Retrieve matching historical weather from Open-Meteo for the same time period.

In [ ]:
start_date = (datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)).date()
end_date = (datetime.now(timezone.utc) - timedelta(days=1)).date()

weather_points = fetch_historical(LAT, LON, start_date, end_date)
df_weather = pd.DataFrame([
    {'timestamp': w.timestamp, 'temperature': w.temperature,
     'humidity': w.humidity, 'wind_speed': w.wind_speed}
    for w in weather_points
])
df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'], utc=True)

print(f"Weather data points: {len(df_weather)}")
df_weather.describe()

## 4. Merge & Correlate

Join energy and weather on the nearest hour, then compute correlations.

In [ ]:
# Round timestamps to hour for joining
df_weather['hour'] = df_weather['timestamp'].dt.floor('h')
df_elec_h['hour'] = df_elec_h['timestamp'].dt.floor('h')
df_therm_h['hour'] = df_therm_h['timestamp'].dt.floor('h')

# Merge all three
df_merged = (df_weather
    .merge(df_elec_h[['hour', 'elec_kwh']], on='hour', how='inner')
    .merge(df_therm_h[['hour', 'therm_kwh']], on='hour', how='inner')
)
print(f"Merged data points: {len(df_merged)}")

# Correlation matrix
corr_cols = ['temperature', 'humidity', 'wind_speed', 'elec_kwh', 'therm_kwh']
corr = df_merged[corr_cols].corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                title='Correlation Matrix: Weather vs Energy',
                zmin=-1, zmax=1)
fig.update_layout(height=450)
fig.show()

## 5. Scatter Plots with Regression

Temperature vs energy consumption with linear trend lines.

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Temperature vs Electrical (kWh)',
                                    'Temperature vs Thermal (kWh)'))

fig.add_trace(go.Scatter(x=df_merged['temperature'], y=df_merged['elec_kwh'],
              mode='markers', marker=dict(size=3, opacity=0.3),
              name='Electrical'), row=1, col=1)

fig.add_trace(go.Scatter(x=df_merged['temperature'], y=df_merged['therm_kwh'],
              mode='markers', marker=dict(size=3, opacity=0.3, color='orange'),
              name='Thermal'), row=1, col=2)

# Add trendlines
for col_idx, (y_col, color) in enumerate(
        [('elec_kwh', 'blue'), ('therm_kwh', 'orange')], 1):
    z = np.polyfit(df_merged['temperature'], df_merged[y_col], 1)
    x_line = np.linspace(df_merged['temperature'].min(), df_merged['temperature'].max(), 100)
    fig.add_trace(go.Scatter(x=x_line, y=np.polyval(z, x_line),
                  mode='lines', line=dict(color=color, width=2),
                  name=f'Trend ({y_col})'), row=1, col=col_idx)

fig.update_layout(height=450, title='Temperature vs Heat Pump Energy')
fig.update_xaxes(title_text='Temperature (°C)')
fig.update_yaxes(title_text='kWh/hour')
fig.show()

## 6. Time-of-Day Energy Pattern

Do energy consumption patterns differ by time of day at different temperatures?

In [ ]:
df_merged['hour_of_day'] = df_merged['hour'].dt.hour
df_merged['temp_band'] = pd.cut(df_merged['temperature'],
                                 bins=[-20, 0, 10, 20, 40],
                                 labels=['< 0°C', '0–10°C', '10–20°C', '> 20°C'])

hourly_avg = (df_merged.groupby(['hour_of_day', 'temp_band'])
              .agg(elec_kwh=('elec_kwh', 'mean'),
                   therm_kwh=('therm_kwh', 'mean'))
              .reset_index())

fig = px.line(hourly_avg, x='hour_of_day', y='elec_kwh', color='temp_band',
              title='Avg Hourly Electrical Energy by Temperature Band',
              labels={'hour_of_day': 'Hour of Day', 'elec_kwh': 'Avg kWh',
                      'temp_band': 'Outdoor Temp'})
fig.update_layout(height=400)
fig.show()